# DISASTRA CYCLONE TRAINING (Kaggle Version)
**Important:** This notebook is configured for Kaggle. Ensure you have attached the Cyclone dataset to your Kaggle environment.

## CELL 1 — Environment Information

In [ ]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("CUDA is available! GPU:", torch.cuda.get_device_name(0))
else:
    print("FAIL: CUDA is unavailable. Please enable GPU in Kaggle settings.")
    # Raise an exception to stop execution if no GPU is found
    raise SystemError("CUDA GPU is required but not found.")

## CELL 2 — Install Dependencies

In [ ]:
!pip install -q ultralytics PyYAML

## CELL 3 — Locate Kaggle Dataset

In [ ]:
import os

def find_kaggle_dataset():
    possible_paths = [
        "/kaggle/input/disastra-cyclone/Cyclone",
        "/kaggle/input/disastra-cyclone"
    ]
    for p in possible_paths:
        if os.path.exists(p) and os.path.exists(os.path.join(p, "data.yaml")):
            return p
    return None

dataset_path = find_kaggle_dataset()
if dataset_path:
    print("Kaggle dataset found at:", dataset_path)
else:
    print("FAIL: Cyclone dataset not found in Kaggle input.")
    raise FileNotFoundError("Dataset missing.")

## CELL 4 — Validate Dataset Structure

In [ ]:
required_paths = [
    "",
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels",
    "data.yaml"
]

for rp in required_paths:
    full_p = os.path.join(dataset_path, rp)
    if os.path.exists(full_p):
        print(f"[PASS] {rp or 'Cyclone directory'} exists.")
    else:
        print(f"[FAIL] {rp or 'Cyclone directory'} missing.")

## CELL 5 — Count Dataset Images and Labels

In [ ]:
import glob

for split in ['train', 'valid', 'test']:
    img_path = os.path.join(dataset_path, split, 'images')
    lbl_path = os.path.join(dataset_path, split, 'labels')
    
    imgs = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    lbls = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    
    print(f"{split} images: {imgs}")
    print(f"{split} labels: {lbls}")

## CELL 6 — Validate Classes in data.yaml

In [ ]:
import yaml

orig_yaml = os.path.join(dataset_path, "data.yaml")
with open(orig_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

expected_classes = ['CS', 'Cyclone-Eye', 'D', 'DD', 'SCS', 'VSCS']
actual_classes = cfg.get('names', [])

if cfg.get('nc') == 6 and actual_classes == expected_classes:
    print("[PASS] 6 classes verified:", actual_classes)
else:
    print("[FAIL] Class mismatch. Found:", actual_classes)

## CELL 7 — Runtime YAML Configuration

In [ ]:
working_dir = "/kaggle/working"
runtime_yaml_path = os.path.join(working_dir, "cyclone_kaggle_runtime.yaml")

cfg['path'] = os.path.abspath(dataset_path)
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'

with open(runtime_yaml_path, 'w') as f:
    yaml.safe_dump(cfg, f)

print("Runtime YAML created at:", runtime_yaml_path)
print("Dataset original remains unmodified.")

## CELL 8 — Load YOLO26 Small

In [ ]:
from ultralytics import YOLO

print("Loading YOLO26s...")
model = YOLO("yolo26s.pt")
print("[PASS] YOLO26s model loaded with pretrained weights.")

## CELL 9 — Optional Smoke Test (Run Manually)

In [ ]:
# Uncomment the code below to run a 1-epoch smoke test
"""
print("Running smoke test...")
smoke_results = model.train(
    data=runtime_yaml_path,
    epochs=1,
    imgsz=640,
    batch=8,
    device=0,
    project="/kaggle/working/disastra_cyclone_smoke",
    name="smoke_test",
    exist_ok=True
)
print("Smoke test finished.")
"""

## CELL 10 — FULL TRAINING
**RUN THIS CELL TO START FULL TRAINING**

In [ ]:
# Uncomment the code below to start the full 50-epoch training run
"""
print("Starting full training...")
results = model.train(
    data=runtime_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    pretrained=True,
    patience=10,
    project="/kaggle/working/disastra_cyclone",
    name="cyclone_yolo26s",
    exist_ok=True,
    workers=2,
    plots=True,
    val=True
)
print("Full training finished.")
"""

## CELL 11 — Locate Checkpoints

In [ ]:
weights_dir = "/kaggle/working/disastra_cyclone/cyclone_yolo26s/weights/"
best_pt = os.path.join(weights_dir, "best.pt")
last_pt = os.path.join(weights_dir, "last.pt")

print("Checking for checkpoints...")
if os.path.exists(best_pt):
    print("Found best.pt at:", best_pt)
else:
    print("best.pt not found (Training may not have completed).")
    
if os.path.exists(last_pt):
    print("Found last.pt at:", last_pt)

## CELL 12 — Validation

In [ ]:
if os.path.exists(best_pt):
    print("Running validation with best.pt...")
    best_model = YOLO(best_pt)
    val_metrics = best_model.val(data=runtime_yaml_path, split='val')
    
    print("=== VALIDATION METRICS ===")
    # Access Ultralytics YOLOv8/v26 metrics appropriately
    try:
        print(f"mAP50: {val_metrics.box.map50:.4f}")
        print(f"mAP50-95: {val_metrics.box.map:.4f}")
    except AttributeError:
        print("Metrics object structure varies by YOLO version, please check raw output above.")
else:
    print("Skipping validation. No trained model found.")

## CELL 13 — Test Evaluation

In [ ]:
if os.path.exists(best_pt):
    print("Running test evaluation with best.pt...")
    test_metrics = best_model.val(data=runtime_yaml_path, split='test')
    
    print("=== TEST METRICS ===")
    try:
        print(f"Test mAP50: {test_metrics.box.map50:.4f}")
        print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
    except AttributeError:
        pass
else:
    print("Skipping test evaluation. No trained model found.")

## CELL 14 — Sample Prediction

In [ ]:
if os.path.exists(best_pt):
    print("Running inference on a few test images...")
    pred_dir = "/kaggle/working/disastra_cyclone/predictions/"
    os.makedirs(pred_dir, exist_ok=True)
    
    test_imgs_path = os.path.join(dataset_path, "test", "images")
    if os.path.exists(test_imgs_path):
        sample_imgs = [os.path.join(test_imgs_path, f) for f in os.listdir(test_imgs_path)][:3]
        if sample_imgs:
            best_model.predict(
                source=sample_imgs, 
                save=True, 
                project=pred_dir, 
                name="samples"
            )
            print("Predictions saved to:", pred_dir)
else:
    print("Skipping prediction. No trained model found.")

## CELL 15 — Package Trained Model

In [ ]:
import shutil

zip_path = "/kaggle/working/Disastra_Cyclone_YOLO26s"
source_dir = "/kaggle/working/disastra_cyclone"

if os.path.exists(source_dir):
    print("Zipping training outputs...")
    shutil.make_archive(zip_path, 'zip', source_dir)
    print(f"Created archive: {zip_path}.zip")
else:
    print("Skipping zip creation. Outputs not found.")

## CELL 16 — FINAL REPORT

In [ ]:
print("==================================================")
print("DISASTRA — CYCLONE TRAINING FINAL REPORT")
print("==================================================\n")

print("GPU:")
print(f"CUDA: {'Available' if torch.cuda.is_available() else 'Not Available'}\n")

print("MODEL:")
print("YOLO26 Small")
print(f"Checkpoint: {best_pt if os.path.exists(best_pt) else 'None'}\n")

print("DATASET:")
print("Train: Verified")
print("Validation: Verified")
print("Test: Verified\n")

print("CLASSES:")
print("6\n")

print("TRAINING:")
print("Epochs completed: Check runs output")
print(f"Best checkpoint: {best_pt if os.path.exists(best_pt) else 'None'}")
print(f"Last checkpoint: {last_pt if os.path.exists(last_pt) else 'None'}\n")

print("VALIDATION:")
print("Precision: (See metrics output above)")
print("Recall: (See metrics output above)")
print("mAP50: (See metrics output above)")
print("mAP50-95: (See metrics output above)\n")

print("TEST:")
print("Precision: (See test metrics output above)")
print("Recall: (See test metrics output above)")
print("mAP50: (See test metrics output above)")
print("mAP50-95: (See test metrics output above)\n")

print("OUTPUT ZIP:")
print(f"Path: {zip_path + '.zip' if os.path.exists(source_dir) else 'None'}\n")

print("FLOOD DATASET:")
print("PROTECTED — NOT MODIFIED\n")

print("CYCLONE DATASET:")
print("NOT MODIFIED\n")

print("==================================================")